In [0]:
"""Bronze layer: load all 5 MaStR CSVs as Delta tables.

Auto Loader reads CSVs from the landing Volume, infers schema,
and writes Delta tables. Re-running this notebook is safe — Auto Loader
remembers what files it has processed via the checkpoint location.

When a new snapshot lands in raw_files/landing/mastr/<new-date>/,
just re-run this notebook to incrementally load it.
"""
from pyspark.sql.functions import current_timestamp, col


LANDING_BASE = "/Volumes/eeg_dev/raw_files/landing/mastr"
SCHEMA_BASE = "/Volumes/eeg_dev/raw_files/landing/_schemas/mastr"

# DEBUG: only ingest hydro for fast iteration. Set to False before final run.
DEBUG_HYDRO_ONLY = False

CATEGORIES = [
    ("solar", "bnetza_mastr_solar_raw.csv"),
    ("wind", "bnetza_mastr_wind_raw.csv"),
    ("biomass", "bnetza_mastr_biomass_raw.csv"),
    ("hydro", "bnetza_mastr_hydro_raw.csv"),
    ("market_actors", "bnetza_mastr_market_actors_raw.csv"),
]

if DEBUG_HYDRO_ONLY:
    CATEGORIES = [c for c in CATEGORIES if c[0] == "hydro"]


def ingest(category: str, filename: str) -> None:
    table = f"eeg_dev.bronze.mastr_{category}"
    schema_loc = f"{SCHEMA_BASE}/{category}"

    print(f"Ingesting {category} -> {table}")

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_loc)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaHints",
            "Gemeindeschluessel string, "
            "Postleitzahl string, "
            "Bruttoleistung double, "
            "Nettonennleistung double"
        )
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(f"{LANDING_BASE}/*/{filename}")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )

    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", f"{schema_loc}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)   # Process all available files, then stop
        .toTable(table)
    )

    query.awaitTermination()
    print(f"  Done: {category}")


for category, filename in CATEGORIES:
    ingest(category, filename)

print("\nAll bronze tables loaded.")

In [0]:
dbutils.fs.rm("/Volumes/eeg_dev/raw_files/landing/_schemas/mastr", recurse=True)
print("Cleared Auto Loader state.")